# CPTS 440 — Chess Agent Demo

This notebook demonstrates our depth-limited minimax search engine with material evaluation.

**What you'll see:**
1. Engine overview, having it choose moves and reporting search statistics
2. Tactical puzzle solving (mate-in-1, winning material)
3. Depth comparison, how deeper search finds better moves at higher cost
4. A full AI-vs-AI game with move-by-move visualization

In [ ]:
import chess
import chess.svg
from IPython.display import SVG, display, HTML

from src import engine
from src.engine import play_game

---
## 1. Engine Overview

Our agent uses **depth-limited minimax** search with a **material-based evaluation function**.

- **Search**: Explores the game tree to a fixed depth, alternating between maximizing (White) and minimizing (Black) players
- **Evaluation**: Scores positions in centipawns based on material balance (P=100, N=320, B=330, R=500, Q=900)
- **Move ordering**: Captures and checks are searched first to improve pruning in future iterations

Let's give the engine a position and see what it finds.

In [ ]:
board = chess.Board()

result = engine.choose_move(board, depth=2)

print(f"Best move : {result.move}")
print(f"Eval score: {result.score:+.0f} centipawns")
print(f"Nodes     : {result.nodes:,}")
print(f"Depth     : {result.depth} plies")

In [ ]:
# Show the board BEFORE and AFTER the engine's chosen move, side by side
before_svg = chess.svg.board(
    board, size=350,
    arrows=[chess.svg.Arrow(result.move.from_square, result.move.to_square, color="#cc220080")],
)

board_after = board.copy()
board_after.push(result.move)
after_svg = chess.svg.board(
    board_after, size=350,
    fill={result.move.to_square: "#88cc44"},
)

display(HTML(f"""
<div style='display:flex; gap:2em; align-items:center'>
  <div><h4>Before</h4>{before_svg}</div>
  <div style='font-size:2em'>→</div>
  <div><h4>After {result.move}</h4>{after_svg}</div>
</div>
"""))

---
## 2. Tactical Puzzles

Can the engine solve simple tactics? Let's test it on known positions.

In [ ]:
def show_puzzle(title: str, fen: str, expected_uci: str, depth: int = 3) -> None:
    """Display a puzzle position, let the engine solve it, and show the result."""
    board = chess.Board(fen)
    result = engine.choose_move(board, depth=depth)

    found = result.move.uci() if result.move else "(none)"
    match = "✅" if found == expected_uci else "❌"

    board_after = board.copy()
    if result.move:
        board_after.push(result.move)

    before_svg = chess.svg.board(
        board, size=300,
        arrows=(
            [chess.svg.Arrow(result.move.from_square, result.move.to_square, color="#cc220080")]
            if result.move else []
        ),
    )
    after_svg = chess.svg.board(board_after, size=300)

    display(HTML(f"""
    <h3>{title} {match}</h3>
    <p>Engine chose <b>{found}</b> (expected <b>{expected_uci}</b>)
    — eval: {result.score:+.0f} cp, nodes: {result.nodes:,}, depth: {result.depth}</p>
    <div style='display:flex; gap:1.5em; align-items:center'>
      <div>{before_svg}</div>
      <div style='font-size:1.5em'>→</div>
      <div>{after_svg}</div>
    </div>
    """))

In [ ]:
# Puzzle 1: White to move with mate in 1 by Qh5#
show_puzzle(
    title="Mate in 1",
    fen="rnbqkbnr/pppp1ppp/8/4p3/6P1/5P2/PPPPP2P/RNBQKBNR b KQkq - 0 2",
    expected_uci="d8h4",
    depth=3,
)

In [ ]:
# Puzzle 2: White to move with win by capturing Black's hanging queen
show_puzzle(
    title="Win the queen",
    fen="r1bqkbnr/pppppppp/2n5/8/3NP3/8/PPP2PPP/RNBQKB1R w KQkq - 1 3",
    expected_uci="d4c6",
    depth=2,
)

In [ ]:
# Puzzle 3: White to move with back-rank mate by Qe8# (rook guards, king trapped)
show_puzzle(
    title="Back-rank mate",
    fen="6k1/5ppp/8/8/8/8/1Q3PPP/6K1 w - - 0 1",
    expected_uci="b2b8",
    depth=3,
)

---
## 3. Depth Comparison

How does increasing search depth affect the engine's choice and the cost (nodes searched)?

We run the same position at depth 1, 2, and 3 and compare.

In [ ]:
fen = "r1bqkb1r/pppppppp/2n2n2/8/4P3/5N2/PPPP1PPP/RNBQKB1R w KQkq - 2 3"
board = chess.Board(fen)

print(f"Position: {fen}\n")
print(f"{'Depth':<8} {'Best Move':<12} {'Score (cp)':<12} {'Nodes':>10}")
print("-" * 46)

for d in range(1, 4):
    result = engine.choose_move(board, depth=d)
    move_str = result.move.uci() if result.move else "(none)"
    print(f"{d:<8} {move_str:<12} {result.score:+<12.0f} {result.nodes:>10,}")

SVG(chess.svg.board(board, size=350))

Notice how:
- **Node count grows exponentially** with depth (branching factor ~30–35 in the middlegame)
- **Deeper search can change the best move** as the engine sees further consequences
- This motivates alpha-beta pruning (Week 3) to search deeper within the same node budget

---
## 4. AI vs. AI Game

Let's run a full game where the engine plays both sides, then visualize the result.

In [ ]:
game = play_game(depth=2, max_moves=80)

print(f"Game result : {game.result}")
print(f"Total plies : {len(game.plies)}")
print(f"Total nodes : {sum(p.nodes for p in game.plies):,}")

In [ ]:
# Show the evaluation score over the course of the game
print(f"{'Ply':<6} {'Move':<10} {'Score (cp)':>10} {'Nodes':>8}  {'Bar'}")
print("-" * 60)

for i, ply in enumerate(game.plies):
    # Simple text bar chart: each █ = 50 centipawns
    bar_len = int(ply.score / 50)
    if bar_len >= 0:
        bar = " " * 20 + "█" * min(bar_len, 20)
    else:
        filled = min(abs(bar_len), 20)
        bar = " " * (20 - filled) + "█" * filled
    print(f"{i+1:<6} {ply.move_uci:<10} {ply.score:>+10.0f} {ply.nodes:>8,}  |{bar}|")

In [ ]:
# Visualize key moments: first move, a midgame position, and the final position
snapshots = []
indices = [0, len(game.plies) // 2, len(game.plies) - 1]

for idx in indices:
    ply = game.plies[idx]
    b = chess.Board(ply.fen_before)
    move = chess.Move.from_uci(ply.move_uci)
    svg = chess.svg.board(
        b, size=280,
        arrows=[chess.svg.Arrow(move.from_square, move.to_square, color="#cc220080")],
    )
    snapshots.append(f"""
    <div style='text-align:center'>
      <h4>Ply {idx + 1}: {ply.move_uci}</h4>
      <p>Eval: {ply.score:+.0f} cp</p>
      {svg}
    </div>
    """)

display(HTML(f"<div style='display:flex; gap:1em; flex-wrap:wrap'>{''.join(snapshots)}</div>"))

In [ ]:
# Show the final board position
final_board = chess.Board(game.final_fen)
display(HTML(f"<h3>Final Position — Result: {game.result}</h3>"))
SVG(chess.svg.board(final_board, size=400))

---
### Interactive HTML Replay

Export the game above to a standalone HTML file with play/pause controls.
Open `game_replay.html` in any browser.

> Use the command `explorer.exe game_replay.html` in the terminal.

In [ ]:
from src.viz import export_game_html

out = export_game_html(game, "game_replay.html")
print(f"Replay saved to: {out}")

---
## Section 5 — Positional Evaluation (PST Comparison)

Piece-square tables (PSTs) assign a positional bonus to each piece based on which square it occupies.
A centralized knight gets up to +20 cp more than a rim knight.
A castled king scores +30 cp vs 0 cp for a king stuck in the centre.

This section shows:
- How the same material counts differently with vs without PSTs
- Which moves the engine **prefers** because of positional bonuses
- A visual heatmap of the PST values for each piece type

In [ ]:
from src import eval as ev

# Positions that highlight the PST effect
positions = [
    ("Knight center (e4) vs rim (a1) — material equal",
     "4k3/8/8/8/4N3/8/7P/4K3 w - - 0 1",
     "4k3/8/8/8/8/8/7P/N3K3 w - - 0 1"),
    ("Pawn advanced (e4) vs retreated (e2) — material equal",
     "4k3/8/8/8/4P3/8/8/4K3 w - - 0 1",
     "4k3/8/8/8/8/8/4P3/4K3 w - - 0 1"),
    ("King castled (g1) vs exposed (e1) — material equal",
     "4k3/8/8/8/8/8/7P/6K1 w - - 0 1",
     "4k3/8/8/8/8/8/7P/4K3 w - - 0 1"),
]

header = f"{'Scenario':<45} {'Pos A (no PST)':>14} {'Pos A (+PST)':>14} {'Pos B (no PST)':>14} {'Pos B (+PST)':>14} {'PST delta':>10}"
print(header)
print("-" * len(header))
for label, fen_a, fen_b in positions:
    ba, bb = chess.Board(fen_a), chess.Board(fen_b)
    a_raw = ev.evaluate(ba, use_pst=False)
    a_pst = ev.evaluate(ba, use_pst=True)
    b_raw = ev.evaluate(bb, use_pst=False)
    b_pst = ev.evaluate(bb, use_pst=True)
    delta = a_pst - b_pst
    print(f"{label:<45} {a_raw:>+14.0f} {a_pst:>+14.0f} {b_raw:>+14.0f} {b_pst:>+14.0f} {delta:>+10.0f}")

In [ ]:
from src.search import choose_move

# Does PST change the first move the engine picks from the opening position?
# We compare choose_move() (which uses PSTs via eval.evaluate) with a patched
# call that forces use_pst=False by temporarily runtime patch to evaluate.

import types, functools

def _choose_move_no_pst(board, *, depth=2):
    """Run choose_move with PSTs disabled by patching evaluate temporarily."""
    original = ev.evaluate
    ev.evaluate = functools.partial(original, use_pst=False)
    try:
        result = choose_move(board, depth=depth)
    finally:
        ev.evaluate = original
    return result

board = chess.Board()
res_pst  = choose_move(board, depth=2)
res_raw  = _choose_move_no_pst(board, depth=2)

print(f"{'Mode':<12} {'Best move':>10} {'Score':>8} {'Nodes':>8}")
print("-" * 42)
print(f"{'With PST':<12} {str(res_pst.move):>10} {res_pst.score:>+8.1f} {res_pst.nodes:>8}")
print(f"{'No PST':<12} {str(res_raw.move):>10} {res_raw.score:>+8.1f} {res_raw.nodes:>8}")

if res_pst.move != res_raw.move:
    print(f"\n  PSTs changed the engine's choice: {res_raw.move} → {res_pst.move}")
else:
    print(f"\n  Both modes agree on best move: {res_pst.move}")

# Show the board after the PST-preferred move
board_after = chess.Board()
board_after.push(res_pst.move)
display(HTML("<b>Position after PST-preferred first move:</b>"))
display(SVG(chess.svg.board(board_after, size=350, lastmove=res_pst.move)))

In [ ]:
from src.eval import PST_PAWN, PST_KNIGHT, PST_BISHOP, PST_ROOK, PST_QUEEN, PST_KING_MIDDLEGAME

# ASCII heat-map helper
def _heatmap(table: list[int], title: str) -> None:
    """Print a coloured ASCII grid (rank 8 at top, file a at left)."""
    lo, hi = min(table), max(table)
    span = max(hi - lo, 1)
    COLS = ["\033[38;5;21m", "\033[38;5;39m", "\033[38;5;82m",
            "\033[38;5;226m", "\033[38;5;208m", "\033[38;5;196m"]
    RESET = "\033[0m"
    print(f"\n  {title}")
    print("   a    b    c    d    e    f    g    h")
    for rank in range(7, -1, -1):          # rank 8 → rank 1
        row = ""
        for file in range(8):              # file a → h
            sq = rank * 8 + file
            val = table[sq]
            idx = int((val - lo) / span * (len(COLS) - 1))
            row += f"{COLS[idx]}{val:+4d}{RESET} "
        print(f"{rank+1}  {row}")

tables = [
    (PST_PAWN,            "PAWN"),
    (PST_KNIGHT,          "KNIGHT"),
    (PST_BISHOP,          "BISHOP"),
    (PST_ROOK,            "ROOK"),
    (PST_QUEEN,           "QUEEN"),
    (PST_KING_MIDDLEGAME, "KING (middlegame)"),
]
for tbl, name in tables:
    _heatmap(tbl, name)